# Processing Textual Data - Exercise 1
## Příprava a čištění textových dat (IMDb Reviews)

**Zadání cvičení:**
1. Stáhněte datový soubor pojmenovaný `imdb_reviews.csv`.
2. Načtěte data ze souboru do proměnné `imdb_reviews` pomocí knihovny **Pandas**.
3. Napište funkci, která jako argument přijme text recenze a vrátí text v očištěné podobě:
   - Všechna slova budou složena z malých písmen (*lowercased*),
   - V textu zůstanou pouze písmena (*použijte regulární výraz*),
   - Z textu budou odstraněna stop-slova (*jakýmkoliv probraným způsobem*).
4. Do DataFrame přidejte sloupec `review_cleaned` s výsledkem volání funkce z kroku 3 na sloupec obsahující texty recenzí (pomocí `.apply()`).
5. Uložte DataFrame do souboru `.csv` jako `imdb_reviews_preprocessed_1.csv`.

In [1]:
import re
import pandas as pd
import nltk
from nltk.corpus import stopwords

# Stáhneme korpus stop-slov NLTK
nltk.download('stopwords', quiet=True)
stop_words = set(stopwords.words('english'))
print(f'NLTK stop-words načteny, celkem {len(stop_words)} slov.')

### Krok 1 & 2: Načtení datového souboru do proměnné `imdb_reviews`

In [2]:
# Načtení datasetu IMDb recenzí
imdb_reviews = pd.read_csv('data/imdb_reviews.csv')
print(f'Rozměry načteného DataFrame: {imdb_reviews.shape}')
print('\nPřehled sloupců a datových typů:')
print(imdb_reviews.info())
imdb_reviews.head()

### Krok 3: Definice čistící funkce `clean_review`
Funkce provádí:
1. Převod na malá písmena (`.lower()`)
2. Ponechání pouze písmen pomocí regulárního výrazu (`re.sub(r'[^a-zA-Z\s]', ' ', text)`)
3. Odstranění stop-slov ze seznamu NLTK

In [3]:
def clean_review(text: str) -> str:
    """
    Očistí text recenze podle zadání:
    - Malá písmena,
    - Pouze písmena (odstranění čísel, HTML tagů, interpunkce),
    - Odstranění anglických stop-slov.
    """
    if not isinstance(text, str):
        return ''
    
    # 1. Lowercase
    text = text.lower()
    
    # 2. Ponechání pouze písmen (nahrazení všeho ostatního mezerou)
    text = re.sub(r'[^a-zA-Z\s]', ' ', text)
    
    # 3. Tokenizace split() a filtrace stop-slov
    words = text.split()
    words_filtered = [w for w in words if w not in stop_words]
    
    return ' '.join(words_filtered)

# Test na první recenzi
sample_raw = imdb_reviews['review'].iloc[0]
sample_clean = clean_review(sample_raw)
print('Původní text (prvních 100 znaků):\n', sample_raw[:100])
print('\nOčištěný text (prvních 100 znaků):\n', sample_clean[:100])

### Krok 4: Aplikace funkce na sloupec `review` pomocí `.apply()`

In [4]:
# Vytvoření nového sloupce review_cleaned
imdb_reviews['review_cleaned'] = imdb_reviews['review'].apply(clean_review)
imdb_reviews[['sentiment', 'review', 'review_cleaned']].head()

### Krok 5: Uložení do souboru `imdb_reviews_preprocessed_1.csv`

In [5]:
output_file = 'data/imdb_reviews_preprocessed_1.csv'
imdb_reviews.to_csv(output_file, index=False)
print(f'DataFrame úspěšně uložen do: {output_file}')

# Kontrolní ověření načtením
df_check = pd.read_csv(output_file)
print(f'Kontrola načtení: tvar {df_check.shape}, sloupce {df_check.columns.tolist()}')